In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project root detection
PROJECT_ROOT = Path.cwd().parents[0] if "notebooks" in str(Path.cwd()) else Path.cwd()

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_FEATURES_DIR = PROJECT_ROOT / "data" / "features"

DATA_PROCESSED_DIR, DATA_FEATURES_DIR


(WindowsPath('c:/Trading/Projects/ES_AI_Project/data/processed'),
 WindowsPath('c:/Trading/Projects/ES_AI_Project/data/features'))

In [14]:
merged_path = DATA_PROCESSED_DIR / "merged_raw.csv"

if not merged_path.exists():
    raise FileNotFoundError(f"merged_raw.csv not found at {merged_path}")

df = pd.read_csv(merged_path)

# Preserve Time column if it exists
if "Time" in df.columns:
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")

df.head()


,Instrument,StrategyName,TradeId,Phase,OffsetFromEntry,BarIndex,Time,IsTradeBar,ShortPatternCandidate,LongPatternCandidate,...,IsLong,IsShort,BarsSinceEntry,Unreal,MAE,MFE,Future5,Future10,Future20,source_file
0,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,50,2026-02-16 09:21:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv
1,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,51,2026-02-16 09:22:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv
2,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,52,2026-02-16 09:23:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv
3,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,53,2026-02-16 09:24:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv
4,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,54,2026-02-16 09:25:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv


In [15]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 28960 entries, 0 to 28959
Data columns (total 89 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Instrument             28960 non-null  str           
 1   StrategyName           28960 non-null  str           
 2   TradeId                0 non-null      float64       
 3   Phase                  28960 non-null  str           
 4   OffsetFromEntry        28960 non-null  int64         
 5   BarIndex               28960 non-null  int64         
 6   Time                   28960 non-null  datetime64[ns]
 7   IsTradeBar             28960 non-null  int64         
 8   ShortPatternCandidate  28960 non-null  int64         
 9   LongPatternCandidate   28960 non-null  int64         
 10  ShortSuccessLabel      28960 non-null  int64         
 11  LongSuccessLabel       28960 non-null  int64         
 12  Close                  28960 non-null  float64       
 13  Open        

In [16]:
# Drop rows missing essential OHLC data
df = df.dropna(subset=["Open", "High", "Low", "Close"])

# Convert Time column if needed
if "Time" in df.columns:
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")

df.head()


,Instrument,StrategyName,TradeId,Phase,OffsetFromEntry,BarIndex,Time,IsTradeBar,ShortPatternCandidate,LongPatternCandidate,...,IsLong,IsShort,BarsSinceEntry,Unreal,MAE,MFE,Future5,Future10,Future20,source_file
0,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,50,2026-02-16 09:21:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv
1,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,51,2026-02-16 09:22:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv
2,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,52,2026-02-16 09:23:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv
3,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,53,2026-02-16 09:24:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv
4,ES 06-26,ES_AI_Swing_HMA_BB,NaN,Context,0,54,2026-02-16 09:25:00,0,0,0,...,0,0,0,0.0,0,0,0,0,0,20260519.csv


In [17]:
FEATURE_COLUMNS = [
    "Close","Open","High","Low","Body","UpperWick","LowerWick","Range",
    "C1","H1","L1","B1","C2","H2","L2","B2","C3","H3","L3","B3","C4","H4","L4","B4","C5","H5","L5","B5",
    "EMA9","EMA21","EMA50","EMA200","FastSlope","SlopeAccel","TrendUp","TrendDown",
    "DistEMA200","Mom","ADX","ATR","BBWidth",
    "Vol","VolSMA","VolNorm","VolPressure",
    "LocalHigh10","LocalLow10","DistLocalHigh10","DistLocalLow10",
    "SweepUp","SweepDown",
    "IsTradeBar","IsLong","IsShort","BarsSinceEntry"
]

len(FEATURE_COLUMNS)


55

In [18]:
for col in FEATURE_COLUMNS:
    if col not in df.columns:
        print(f"Missing column: {col} — creating placeholder")
        df[col] = 0.0


In [19]:
feature_df = df[FEATURE_COLUMNS].copy()
keep_cols = ["Time"] + FEATURE_COLUMNS
feature_df = df[keep_cols].copy()

feature_df.head()


,Time,Close,Open,High,Low,Body,UpperWick,LowerWick,Range,C1,...,LocalHigh10,LocalLow10,DistLocalHigh10,DistLocalLow10,SweepUp,SweepDown,IsTradeBar,IsLong,IsShort,BarsSinceEntry
0,2026-02-16 09:21:00,6906.75,6906.25,6907.25,6905.50,0.50,0.50,0.75,1.75,6906.25,...,6912.00,6905.00,-21,7,0,0,0,0,0,0
1,2026-02-16 09:22:00,6904.50,6906.50,6906.75,6901.75,-2.00,0.25,2.75,5.00,6906.75,...,6912.00,6901.75,-30,11,0,0,0,0,0,0
2,2026-02-16 09:23:00,6903.00,6904.50,6904.50,6900.75,-1.50,0.00,2.25,3.75,6904.50,...,6910.75,6900.75,-31,9,0,0,0,0,0,0
3,2026-02-16 09:24:00,6904.25,6903.00,6904.50,6903.00,1.25,0.25,0.00,1.50,6903.00,...,6909.75,6900.75,-22,14,0,0,0,0,0,0
4,2026-02-16 09:25:00,6903.00,6904.25,6904.50,6902.50,-1.25,0.25,0.50,2.00,6904.25,...,6909.75,6900.75,-27,9,0,0,0,0,0,0


In [20]:
feature_df.isna().sum().sort_values(ascending=False).head(20)


Time         0
Close        0
Open         0
High         0
Low          0
Body         0
UpperWick    0
LowerWick    0
Range        0
C1           0
H1           0
L1           0
B1           0
C2           0
H2           0
L2           0
B2           0
C3           0
H3           0
L3           0
dtype: int64

In [21]:
feature_df = feature_df.fillna(0)


In [23]:
DATA_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

out_path = DATA_FEATURES_DIR / "features.csv"
feature_df.to_csv(out_path, index=False)

out_path


WindowsPath('c:/Trading/Projects/ES_AI_Project/data/features/features.csv')

In [24]:
loaded = pd.read_csv(out_path)
loaded.head()


,Time,Close,Open,High,Low,Body,UpperWick,LowerWick,Range,C1,...,LocalHigh10,LocalLow10,DistLocalHigh10,DistLocalLow10,SweepUp,SweepDown,IsTradeBar,IsLong,IsShort,BarsSinceEntry
0,2026-02-16 09:21:00,6906.75,6906.25,6907.25,6905.50,0.50,0.50,0.75,1.75,6906.25,...,6912.00,6905.00,-21,7,0,0,0,0,0,0
1,2026-02-16 09:22:00,6904.50,6906.50,6906.75,6901.75,-2.00,0.25,2.75,5.00,6906.75,...,6912.00,6901.75,-30,11,0,0,0,0,0,0
2,2026-02-16 09:23:00,6903.00,6904.50,6904.50,6900.75,-1.50,0.00,2.25,3.75,6904.50,...,6910.75,6900.75,-31,9,0,0,0,0,0,0
3,2026-02-16 09:24:00,6904.25,6903.00,6904.50,6903.00,1.25,0.25,0.00,1.50,6903.00,...,6909.75,6900.75,-22,14,0,0,0,0,0,0
4,2026-02-16 09:25:00,6903.00,6904.25,6904.50,6902.50,-1.25,0.25,0.50,2.00,6904.25,...,6909.75,6900.75,-27,9,0,0,0,0,0,0
